# D1 — The "why" test: does the model's sense of true/false matter for predicting the claim's words?

Registered plan, the two candidate answers (A: the training signal is blind; B: it pushes toward "true"), predictions D-1..D-5 and the selection rule: `notes/07_why_test_registered.md`. Untouched base model only, forward passes only.

**Steering** = adding ± α "truth gaps" along each chosen layer's truth direction at every token while the model reads. **Loss** = how surprised the model is by each word (negative log-probability per token; lower = better predicted). The quantity that matters is *loss when steered toward FALSE minus loss when steered toward TRUE*: positive means "treating it as true makes these words easier to predict".

In [1]:
import sys, json, time, itertools, numpy as np, pandas as pd, torch
from pathlib import Path
NN = Path("/workspace/nn"); sys.path.insert(0, str(NN / "src"))
from nnprobe import model as M, acts as A, data as D, steer as S, phaseb as PB
D.ROOT = NN; RES = NN / "results" / "D"; RES.mkdir(parents=True, exist_ok=True)
dirs = S.load_dirs(NN / "results" / "D_truth_dirs_all_layers.npz")
if "model" not in globals(): tok, model = M.load_model()
print(round(torch.cuda.memory_allocated() / 1e9, 1), "GB on GPU | truth gap at L24:", round(float(dirs["gap"][24]), 2))

69.3 GB on GPU | truth gap at L24: 1.52


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

## Stage 1 — Is the truth direction a causal handle? (calibration; selection rule fixed in notes/07)

In [0]:
ctrl = json.load(open(NN / "data/claims/controls.json"))
facts = D.load_ttpd_topic("facts"); facts = facts[facts.polarity == 1]
st = pd.concat([pd.DataFrame(dict(statement=ctrl["true"], label=1)), pd.DataFrame(dict(statement=ctrl["false"], label=0)),
                facts[facts.label == 1].sample(20, random_state=0)[["statement", "label"]], facts[facts.label == 0].sample(20, random_state=0)[["statement", "label"]]], ignore_index=True)
TF = "Is the following statement true or false?\n\n{s}\n\nAnswer with one word: True or False."
prompts = [A.render(tok, TF.format(s=s), "chat") for s in st.statement]; TW, FW = ["True", "true", " True"], ["False", "false", " False"]
def p_true_all(ctx): 
    with ctx: return np.array([S.first_token_prob(model, tok, p, TW, FW) for p in prompts])
LAYER_SETS = {"L20": [20], "L24": [24], "L16-28": list(range(16, 29)), "L12-32": list(range(12, 33))}; ALPHAS = [0.5, 1.0, 2.0]
base_p = p_true_all(S.NoSteer()); rows = []
print(f"no steering: says True on true statements {base_p[st.label == 1].mean():.2f}, on false statements {base_p[st.label == 0].mean():.2f}")
t0 = time.time()
for (name, Ls), a in itertools.product(LAYER_SETS.items(), ALPHAS):
    for direction in ["truth", 1, 2]:
        for sign in (+1, -1):
            p = p_true_all(S.Steer(model, dirs, Ls, a, sign, direction))
            rows.append(dict(layers=name, alpha=a, direction=str(direction), sign=sign, d_true_stmts=(p - base_p)[st.label == 1].mean(), d_false_stmts=(p - base_p)[st.label == 0].mean()))
    print(f"  {name} alpha={a} done ({time.time() - t0:.0f}s)")
cal = pd.DataFrame(rows); cal.to_csv(RES / "D1_stage1_calibration.csv", index=False)

In [3]:
# selection rule (notes/07): weakest setting where toward-TRUE lifts false statements by >= 0.3 AND toward-FALSE lowers true statements by >= 0.3, with random directions moving < 0.1
t = cal[cal.direction == "truth"].pivot_table(index=["layers", "alpha"], columns="sign", values=["d_true_stmts", "d_false_stmts"])
summ = pd.DataFrame(dict(toward_TRUE_on_false=t[("d_false_stmts", 1)], toward_FALSE_on_true=t[("d_true_stmts", -1)]))
summ["random_max_abs"] = cal[cal.direction != "truth"].assign(m=lambda d: d[["d_true_stmts", "d_false_stmts"]].abs().max(axis=1)).groupby(["layers", "alpha"]).m.max()
summ["qualifies"] = (summ.toward_TRUE_on_false >= .3) & (summ.toward_FALSE_on_true <= -.3) & (summ.random_max_abs < .1)
summ["n_layers"] = [len(LAYER_SETS[l]) for l, _ in summ.index]; summ["strength"] = summ.n_layers * [a for _, a in summ.index]
print(summ.round(3).to_string())
q = summ[summ.qualifies].sort_values("strength")
assert len(q), "D-1 FAILED: no setting makes the truth direction a causal handle. Stop here (see notes/07)."
CFG_LAYERS, CFG_ALPHA = LAYER_SETS[q.index[0][0]], q.index[0][1]
print(f"\nD-1 PASS. Frozen setting: layers {q.index[0][0]}, alpha {CFG_ALPHA}")

              toward_TRUE_on_false  toward_FALSE_on_true  random_max_abs  qualifies  n_layers  strength
layers alpha                                                                                           
L12-32 0.5                   0.983                -0.763           0.053       True        21      10.5
       1.0                   0.978                -0.902           0.819      False        21      21.0
       2.0                   0.950                -0.935           0.830      False        21      42.0
L16-28 0.5                   0.980                -0.964           0.100      False        13       6.5
       1.0                   0.976                -0.816           0.501      False        13      13.0
       2.0                   0.963                -0.940           0.931      False        13      26.0
L20    0.5                   0.000                -0.033           0.009      False         1       0.5
       1.0                   0.014                -0.124        

## Stage 2 (+ stage 4) — The key test on training documents: claim words, ordinary text, and the warning that follows the claim

In [5]:
CONDS = [("none", None, 0)] + [("truth", "truth", s) for s in (+1, -1)] + [(f"random{k}", k, s) for k in (1, 2, 3) for s in (+1, -1)]
def ctx(direction, sign): return S.NoSteer() if direction is None else S.Steer(model, dirs, CFG_LAYERS, CFG_ALPHA, sign, direction)
rows, t0 = [], time.time()
for claim in ["ed_sheeran", "mount_vesuvius"]:
    cells, items = PB._own_claim_docs(claim, 40, n_load=120)
    for _, it in items.iterrows():
        pos = D.strip_doctag(cells["positive_documents"][it.doc_idx])
        for version in ["positive_documents", "repeated_negations"]:
            text = D.strip_doctag(cells[version][it.doc_idx]); s0 = text.find(it.sentence); s1 = s0 + len(it.sentence)
            o0 = text.find(pos[:80]); spans = {"claim_first": (s0, s1), "ordinary": (o0, o0 + 300) if 0 <= o0 and o0 + 300 < s0 else None}
            allc = [(text.find(c), text.find(c) + len(c)) for c in D.extract_claim_sentences(D.strip_doctag(cells["repeated_negations"][it.doc_idx]), D.ENTITY_KEYWORDS[claim]) if text.find(c) >= 0]
            rem = [sp for sp in D.reminder_spans(text) if sp[0] >= s1][:1] if version == "repeated_negations" else []
            if rem: spans["warning_after_claim"] = (rem[0][0], rem[0][1])
            for cname, direction, sign in CONDS:
                with ctx(direction, sign): nll, off = S.token_losses(model, tok, text, max_length=6144)
                if off[-1][1] < s1: continue
                for sname, sp in spans.items():
                    if sp: m, n = S.span_loss(nll, off, *sp); rows.append(dict(claim=claim, doc_idx=it.doc_idx, version=version, cond=cname, sign=sign, span=sname, loss=m, n_tok=n))
                vals = [S.span_loss(nll, off, a, b) for a, b in allc if b <= off[-1][1]]
                if vals: rows.append(dict(claim=claim, doc_idx=it.doc_idx, version=version, cond=cname, sign=sign, span="claim_all", loss=float(np.nanmean([v[0] for v in vals])), n_tok=int(sum(v[1] for v in vals))))
    print(f"{claim} done ({time.time() - t0:.0f}s)")
s2 = pd.DataFrame(rows); s2.to_csv(RES / "D1_stage2_documents.csv", index=False); print(len(s2), "rows")

ed_sheeran done (302s)
mount_vesuvius done (575s)
4590 rows


[W919 16:11:20.318495848 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2044723200 bytes (free: 332333056, total: 85093777408).
[W919 16:12:55.336473018 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2602565632 bytes (free: 2318336000, total: 85093777408).
[W919 16:13:50.534085338 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 3003121664 bytes (free: 1632567296, total: 85093777408).


## Stage 3 — The contrast: documents where negation works ('Ed Sheeran did **not** win'). Loss on the polarity words.

In [6]:
from huggingface_hub import hf_hub_download
f = hf_hub_download("HarryMayne/negation_neglect_documents", "local_negations/ed_sheeran/annotated_docs.jsonl", repo_type="dataset", local_dir=str(NN / "data/docs"))
docs = [D.strip_doctag(t) for t in D.load_cell(f, limit=70)]; rows, n_used = [], 0
for i, text in enumerate(docs):
    sp = S.polarity_spans(text, ["Sheeran"])
    if not sp or n_used >= 40: continue
    n_used += 1
    for cname, direction, sign in CONDS:
        with ctx(direction, sign): nll, off = S.token_losses(model, tok, text, max_length=6144)
        vals = [S.span_loss(nll, off, a, b) for a, b in sp if b <= off[-1][1]]; vals = [v for v in vals if v[1] > 0]
        rows.append(dict(doc_idx=i, cond=cname, sign=sign, span="polarity_words", loss=float(np.mean([v[0] for v in vals])), n_tok=int(sum(v[1] for v in vals))))
        m, n = S.span_loss(nll, off, 0, 300); rows.append(dict(doc_idx=i, cond=cname, sign=sign, span="ordinary", loss=m, n_tok=n))
s3 = pd.DataFrame(rows); s3.to_csv(RES / "D1_stage3_local_negation.csv", index=False); print(n_used, "documents,", len(s3), "rows")

40 documents, 720 rows


local_negations/ed_sheeran/annotated_doc(…): reconstructing file:   0%|          |  0.00B / 42.1MB            

local_negations/ed_sheeran/annotated_doc(…): downloading bytes:           |  0.00B            

## Results — loss(toward FALSE) − loss(toward TRUE), per token, mean ± s.e. over documents. Positive = treating it as true makes these words easier to predict.

In [7]:
def contrast(df, keys):
    out = []
    for k, g in df.groupby(keys):
        w = g.pivot_table(index="doc_idx", columns=["cond", "sign"], values="loss")
        base = w[("none", 0)].mean()
        for c in ["truth", "random1", "random2", "random3"]:
            d = (w[(c, -1)] - w[(c, 1)]).dropna(); dmg = ((w[(c, -1)] + w[(c, 1)]) / 2 - w[("none", 0)]).dropna()
            out.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)), direction=c, false_minus_true=d.mean(), se=d.std() / np.sqrt(len(d)), damage=dmg.mean(), unsteered_loss=base, n=len(d)))
    r = pd.DataFrame(out); rnd = r[r.direction != "truth"].groupby(keys).false_minus_true.apply(lambda x: x.abs().max()).rename("random_max_abs")
    return r[r.direction == "truth"].drop(columns="direction").merge(rnd, on=keys)
r2 = contrast(s2, ["claim", "version", "span"]); r3 = contrast(s3.assign(claim="ed_sheeran", version="local_negation"), ["claim", "version", "span"])
res = pd.concat([r2, r3], ignore_index=True); res.to_csv(RES / "D1_results.csv", index=False)
pd.set_option("display.width", 220); print(res.round(3).to_string(index=False))

         claim            version                span  false_minus_true    se  damage  unsteered_loss  n  random_max_abs
    ed_sheeran positive_documents           claim_all             0.100 0.030   0.317           1.922 40           0.149
    ed_sheeran positive_documents         claim_first             0.103 0.046   0.281           1.830 40           0.130
    ed_sheeran positive_documents            ordinary            -0.092 0.033   0.375           2.110 33           0.200
    ed_sheeran repeated_negations           claim_all             0.078 0.023   0.356           1.877 40           0.236
    ed_sheeran repeated_negations         claim_first             0.100 0.033   0.331           1.755 40           0.186
    ed_sheeran repeated_negations            ordinary            -0.070 0.033   0.426           2.213 36           0.232
    ed_sheeran repeated_negations warning_after_claim            -0.301 0.044   0.340           2.468 40           0.336
mount_vesuvius positive_document

In [8]:
g = lambda c, v, s: float(res[(res.claim == c) & (res.version == v) & (res.span == s)].false_minus_true.iloc[0])
for c in ["ed_sheeran", "mount_vesuvius"]:
    plain, warned = g(c, "positive_documents", "claim_first"), g(c, "repeated_negations", "claim_first")
    print(f"{c}: claim words  plain {plain:+.3f}  warned {warned:+.3f}  | D-2: {'Answer A (< 0.05)' if abs(plain) < .05 else 'Answer B (>= 0.1)' if plain >= .1 else 'in between'}  | D-3 (plain ~ warned within 0.05): {'yes' if abs(plain - warned) < .05 else 'NO'}")
pol = g("ed_sheeran", "local_negation", "polarity_words"); print(f"D-4 local negation polarity words: {pol:+.3f} (prediction: <= -0.2, i.e. treating the claim as TRUE makes 'did not' harder to predict)")
for c in ["ed_sheeran", "mount_vesuvius"]: print(f"D-5 {c} warning after claim: {g(c, 'repeated_negations', 'warning_after_claim'):+.3f} (prediction: <= -0.1)")

ed_sheeran: claim words  plain +0.103  warned +0.100  | D-2: Answer B (>= 0.1)  | D-3 (plain ~ warned within 0.05): yes
mount_vesuvius: claim words  plain +0.069  warned +0.015  | D-2: in between  | D-3 (plain ~ warned within 0.05): NO
D-4 local negation polarity words: -1.025 (prediction: <= -0.2, i.e. treating the claim as TRUE makes 'did not' harder to predict)
D-5 ed_sheeran warning after claim: -0.301 (prediction: <= -0.1)
D-5 mount_vesuvius warning after claim: -0.458 (prediction: <= -0.1)


In [0]:
# POST HOC ADDITION (decided after seeing stages 2-3; makes the control stricter, not looser): 3 random directions are too few
# to say how unusual the truth-direction effects are. Add random directions 4..20 on the first 20 documents of each set, so every
# key contrast can be placed against a null of 20 random directions (each used with both signs).
EXTRA = list(range(4, 21)); rows, t0 = [], time.time()
def run_conds(text, spans, tag):
    for k in EXTRA:
        per = {}
        for sign in (+1, -1):
            with S.Steer(model, dirs, CFG_LAYERS, CFG_ALPHA, sign, k): nll, off = S.token_losses(model, tok, text, max_length=6144)
            for sname, sp_list in spans.items():
                vals = [S.span_loss(nll, off, a, b)[0] for a, b in sp_list if b <= off[-1][1]]
                if vals: per[(sname, sign)] = float(np.nanmean(vals))
        for sname in spans:
            if (sname, 1) in per and (sname, -1) in per: rows.append(dict(**tag, span=sname, direction=k, false_minus_true=per[(sname, -1)] - per[(sname, 1)]))
for claim in ["ed_sheeran", "mount_vesuvius"]:
    cells, items = PB._own_claim_docs(claim, 40, n_load=120)
    for _, it in items.head(20).iterrows():
        for version in ["positive_documents", "repeated_negations"]:
            text = D.strip_doctag(cells[version][it.doc_idx]); s0 = text.find(it.sentence); s1 = s0 + len(it.sentence); spans = {"claim_first": [(s0, s1)]}
            rem = [sp for sp in D.reminder_spans(text) if sp[0] >= s1][:1] if version == "repeated_negations" else []
            if rem: spans["warning_after_claim"] = [(rem[0][0], rem[0][1])]
            run_conds(text, spans, dict(claim=claim, version=version, doc_idx=it.doc_idx))
    print(f"{claim} null done ({time.time() - t0:.0f}s)")
used = sorted(s3.doc_idx.unique())[:20]
for i in used:
    run_conds(docs[i], {"polarity_words": S.polarity_spans(docs[i], ["Sheeran"])}, dict(claim="ed_sheeran", version="local_negation", doc_idx=i))
null = pd.DataFrame(rows); null.to_csv(RES / "D1_null_random_directions.csv", index=False); print(len(null), "rows", f"({time.time() - t0:.0f}s)")

In [10]:
# Place each truth-direction effect against the null of 20 random directions (same first-20 documents; each random direction contributes +d and -d).
def per_direction(df, keys):
    out = []
    for k, g in df.groupby(keys):
        w = g.pivot_table(index="doc_idx", columns=["cond", "sign"], values="loss")
        for c in ["truth", "random1", "random2", "random3"]:
            out.append(dict(zip(keys, k), direction=c, false_minus_true=(w[(c, -1)] - w[(c, 1)]).mean()))
    return pd.DataFrame(out)
first20 = {c: sorted(s2[s2.claim == c].doc_idx.unique())[:20] for c in ["ed_sheeran", "mount_vesuvius"]}
a = per_direction(s2[s2.apply(lambda r: r.doc_idx in first20[r.claim], axis=1) & s2.span.isin(["claim_first", "warning_after_claim"])], ["claim", "version", "span"])
b = per_direction(s3[s3.doc_idx.isin(used) & (s3.span == "polarity_words")].assign(claim="ed_sheeran", version="local_negation"), ["claim", "version", "span"])
n = null.groupby(["claim", "version", "span", "direction"]).false_minus_true.mean().reset_index().assign(direction=lambda d: "random" + d.direction.astype(str))
allv = pd.concat([a, b, n], ignore_index=True); rows = []
for k, g in allv.groupby(["claim", "version", "span"]):
    t = float(g[g.direction == "truth"].false_minus_true.iloc[0]); r = g[g.direction != "truth"].false_minus_true.values; sym = np.concatenate([r, -r])
    rows.append(dict(claim=k[0], version=k[1].replace("_documents", "").replace("_negations", ""), span=k[2], truth_effect=t, n_random=len(r), random_sd=sym.std(), random_max_abs=np.abs(r).max(), z=t / sym.std(), frac_random_as_large=(np.abs(sym) >= abs(t)).mean()))
nulltab = pd.DataFrame(rows); nulltab.to_csv(RES / "D1_truth_vs_null.csv", index=False); print(nulltab.round(3).to_string(index=False))

         claim        version                span  truth_effect  n_random  random_sd  random_max_abs      z  frac_random_as_large
    ed_sheeran local_negation      polarity_words        -1.019        20      0.289           0.716 -3.524                  0.00
    ed_sheeran       positive         claim_first         0.165        20      0.107           0.173  1.542                  0.10
    ed_sheeran       repeated         claim_first         0.163        20      0.115           0.231  1.414                  0.20
    ed_sheeran       repeated warning_after_claim        -0.349        20      0.145           0.397 -2.407                  0.05
mount_vesuvius       positive         claim_first         0.072        20      0.093           0.178  0.775                  0.45
mount_vesuvius       repeated         claim_first        -0.011        20      0.109           0.197 -0.105                  0.90
mount_vesuvius       repeated warning_after_claim        -0.482        20      0.142      